# ARCS 0.6.4: from a case file to impedance

This beginner tutorial follows a 30 mm-radius circular copper pipe. It computes
longitudinal, dipolar and quadrupolar impedance at **100 frequencies from 10 Hz
to 10¹³ Hz**, for a **1.0 m uniform pipe**, with **β = 1** and **roughness disabled**.
The included output is a newly executed example, not an analytical accuracy certificate.

The workflow is **case → boundary meshes → solver → frequency sweep → plots/data**.
A case contains the geometry, material regions, material properties, beam position,
β and mesh settings. A solver prepares the geometry for numerical field calculations.
The sweep solves the synchronous Maxwell problem at each frequency; it is not a
time-domain wake simulation. Wake conversion is a separate step.

Install ARCS from the extracted release folder:

```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install -e ".[gui,cad,notebook]"
python -m jupyterlab
```

On Windows, activate with `.venv\Scripts\activate`. Open this notebook with that
environment's kernel. The tutorial writes its example and results only into
`results/ARCS_workflow_tutorial` beneath the notebook's working directory.
For your own model, replace the example case-file path below with your saved JSON.

This v0.6.5 GUI-material release retains the executed v0.6.4 plots below. The numerical field modules are unchanged. The code version guard is updated, and non-built-in coating properties are defined explicitly.


## 1. Import the tools

The version check avoids silently running a different ARCS installation.
`plot_longitudinal` returns a Matplotlib figure; its length-normalization flag
controls axis labels, not the numerical values.

In [ ]:
from pathlib import Path
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
import arcs
from arcs import load_geometry
from arcs.plotting import plot_longitudinal, plot_transverse

if arcs.__version__ != "0.6.6":
    raise RuntimeError(f"This tutorial targets ARCS 0.6.6; found {arcs.__version__} at {arcs.__file__}")
OUTPUT = Path.cwd() / "results" / "ARCS_workflow_tutorial"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("ARCS version:", arcs.__version__)
print("Imported from:", arcs.__file__)
print("Output directory:", OUTPUT)

## 2. Define and load the case

This writes a small, portable JSON case so the notebook can run without a separate
input download. Cross-section lengths in the dictionary are **millimetres**;
`pipe_length_m`, supplied later, is always **metres**. Conductivity is in S/m.
The vacuum is the first region; the copper exterior is unbounded.
There is one physical boundary here. Coated cases also list each material interface,
its enclosing region and its material properties in the JSON. `make_solver()`
constructs these meshes and materials automatically.

For a GUI-defined case, save its JSON, then use `case = load_geometry("my_pipe.json")`
instead. Loading only a CSV outline is not enough to specify a coated material stack.

In [ ]:
definition = {
    "format": "arcs.geometry/1",
    "name": "Workflow example: 30 mm copper circle",
    "units": "mm",
    "aperture": {"type": "circle", "radius": 30.0},
    "interfaces": [],
    "regions": [
        {"name": "beam_region", "material": "vacuum"},
        {"name": "exterior", "material": "Cu"},
    ],
    "beam": [0.0, 0.0],
    "beta": 1.0,
    "mesh": {
        "minimum_panels": 256,
        "max_panel_length": 1.0,
        "curve_tolerance": 0.01,
        "corner_grading": 2.0,
        "integration_order": 4,
    },
    "materials": {
        "Cu": {"conductivity": 5.8e7, "mu_r": 1.0,
               "epsilon_r": 1.0, "relaxation_time": 0.0},
    },
}
CASE_FILE = OUTPUT / "bare_copper.json"
CASE_FILE.write_text(json.dumps(definition, indent=2) + "\n")
case = load_geometry(CASE_FILE)
print("Loaded:", case.name)
print("beta:", case.beta, "  beam position [m]:", case.beam_position)
print("Roughness:", "disabled" if case.roughness is None else "enabled")

## 3. Inspect the actual mesh before solving

A **panel** is a boundary segment. A **node** is an endpoint; a closed loop with N
straight panels has N distinct nodes. The displayed Gauss samples are additional
points inside each panel: 256 panels × 4 samples = 1024 geometry integration points.
These are not 1024 field unknowns. In this release the GUI/JSON `integration_order`
controls geometry sampling/export; the Maxwell kernels use their own integration
rules. Increasing that GUI value alone does not refine the electromagnetic solve.

There is no mesh across the vacuum area. One boundary mesh is constructed for the
aperture and another for each material interface. The geometry meshes are reused
throughout this frequency sweep; frequency-dependent operators are rebuilt as needed.
The prescribed panel count is a minimum: maximum panel length, curve tolerance and
corner grading can increase it. Corners remain nodes and are not smoothed away.

In [ ]:
report = case.report()
for boundary in report["boundaries"]:
    n = boundary["panels"]
    print(f"{boundary['name']}: {n} panels, {n} nodes, "
          f"{boundary['integration_points']} geometry Gauss points")
    print(f"  actual maximum panel length: {1e3*boundary['maximum_panel_length_m']:.6f} mm")

fig, axes = plt.subplots(1, 3, figsize=(13, 4), layout="constrained")
case.draw(axes[0], show_mesh=False)
axes[0].set_title("Native geometry")
case.draw(axes[1], show_mesh=True)
axes[1].set_title("Boundary panels and nodes")
case.draw(axes[2], show_mesh=True)
axes[2].set(xlim=(29.78, 30.02), ylim=(-3.2, 3.2),
            title="Mesh detail: right wall", aspect="auto")
axes[2].legend().remove()
fig.suptitle("Geometry and discretization are separate views", fontsize=13)
fig.savefig(OUTPUT / "geometry_and_mesh.png", dpi=180)
plt.close(fig)
display(Image(filename=str(OUTPUT / "geometry_and_mesh.png")))

## 4. Prepare the solver and frequency grid

`case.make_solver()` is the usual interface. You do not also need to instantiate
`Metal`, `Wall`, `Layer` or the lower-level `MaxwellSolver` when the case defines
the material stack. The defaults select full Maxwell and **1.0 m pipe totals**.
The explicit length below makes the normalization visible.

For per-unit-length results, use `pipe_length_m=None`. Totals describe a uniform
pipe; changing the length scales the result and does not add entrance/exit fields.

`np.geomspace` specifies frequency sampling only. It does not request more panels.
This 100-point scan is useful to locate features across twelve decades; a narrow
peak or zero crossing may require extra frequency samples around that feature.

In [ ]:
PIPE_LENGTH_M = 1.0
solver = case.make_solver(pipe_length_m=PIPE_LENGTH_M)
frequency = np.geomspace(10.0, 1e13, 100)
print(f"{frequency.size} frequencies: {frequency[0]:g} to {frequency[-1]:g} Hz")
print("Uniform pipe length [m]:", solver.pipe_length_m)
print("Longitudinal units:", solver.longitudinal_units)
print("Dipolar / quadrupolar units:", solver.transverse_coefficient_units)

## 5. Solve the sweep once

Use `solver.longitudinal(frequency)` if you need only longitudinal impedance.
Use `solver.transverse(frequency)` when you need transverse terms too: it **also
returns longitudinal impedance**, so a preceding longitudinal sweep is unnecessary.
The tensors include xx, xy, yx and yy, and a transverse constant term is returned too.
Source/witness derivatives are evaluated analytically; you do not need to run a
sequence of beam offsets to obtain these linear coefficients.

The default component name `wall` means **resistive wall + indirect space charge**,
with direct free-space space charge excluded. To obtain the components together,
replace the next solve by:

```python
components = solver.transverse_components(frequency)
response = components["wall"]
rw = components["resistive_wall"]
image = components["indirect_space_charge"]
```

This example uses β=1. A finite-beta case is selected by changing `beta` in the
case definition before building the solver.

The array API currently loops over frequencies **sequentially**. This cell can take
time; the timing below is the observed runtime for this example and environment.

In [ ]:
started = time.perf_counter()
response = solver.transverse(frequency)
elapsed_s = time.perf_counter() - started
assert np.isfinite(response.longitudinal).all()
assert np.isfinite(response.dipolar).all()
assert np.isfinite(response.quadrupolar).all()
assert np.isfinite(response.constant).all()
print(f"Completed {frequency.size} frequencies in {elapsed_s:.2f} seconds")
print("Longitudinal shape:", response.longitudinal.shape)
print("Dipolar shape:", response.dipolar.shape)
print("Quadrupolar shape:", response.quadrupolar.shape)
print("Transverse constant shape:", response.constant.shape)
print("Component:", response.component)

## 6. Plot signed data and save every tensor entry

Longitudinal is in Ω for the chosen 1.0 m pipe. Dipolar and quadrupolar coefficients
are in Ω/m because they multiply a transverse offset; they are nevertheless already
integrated over the chosen pipe length. Per-length coefficients would instead be Ω/m².
The longitudinal plotting flag must therefore be `per_unit_length=False`.

Signed imaginary values are plotted directly. The transverse overview shows diagonal
x/y coefficients; the CSV also saves off-diagonal entries and the constant term.
The centered circular example is symmetric: its quadrupolar and cross terms are
at the numerical residual level. The bottom axes therefore use a linear scale
and identify these tiny residuals explicitly. Use a noncircular example for a
meaningful quadrupolar validation. The styling changes below affect axes only.

For linear axes, pass `xscale="linear", yscale="linear"` to either plotting function.
This changes the display, not the frequency grid or solver results.

In [ ]:
np.savetxt(
    OUTPUT / "impedance.csv",
    np.column_stack((frequency, response.longitudinal.real, response.longitudinal.imag)),
    delimiter=",", comments="",
    header="frequency_Hz,Re_Z_parallel_Ohm,Im_Z_parallel_Ohm",
)
columns, labels = [frequency], ["frequency_Hz"]
for tensor, kind in ((response.dipolar, "dipolar"), (response.quadrupolar, "quadrupolar")):
    for i, first in enumerate("xy"):
        for j, second in enumerate("xy"):
            z = tensor[:, i, j]
            columns.extend((z.real, z.imag))
            labels.extend((f"Re_{kind}_{first}{second}_Ohm_per_m",
                           f"Im_{kind}_{first}{second}_Ohm_per_m"))
for i, name in enumerate("xy"):
    z = response.constant[:, i]
    columns.extend((z.real, z.imag))
    labels.extend((f"Re_constant_{name}_Ohm", f"Im_constant_{name}_Ohm"))
np.savetxt(OUTPUT / "transverse.csv", np.column_stack(columns),
           delimiter=",", comments="", header=",".join(labels))

fig = plot_longitudinal(
    frequency, response.longitudinal, per_unit_length=False,
    title="ARCS workflow: 30 mm copper circle, 1.0 m pipe, beta = 1",
)
fig.savefig(OUTPUT / "impedance.png", dpi=180)
plt.close(fig)
display(Image(filename=str(OUTPUT / "impedance.png")))

fig = plot_transverse(frequency, response, title="ARCS workflow: circular example")
# Show signed dipolar values across decades without a large empty negative range.
# The circular quadrupolar result is a tiny numerical residual, not a feature.
for j, axis in enumerate(fig.axes[:2]):
    z = response.dipolar[:, j, j]
    axis.set_yscale("symlog", linthresh=0.1)
    axis.set_ylim(-1.0, 1.3*max(z.real.max(), z.imag.max()))
    axis.set_yticks([-1, 0, 1, 10, 100, 1000, 10000])
    axis.set_yticklabels(["-1", "0", "1", "10", "100", "1000", "10000"])
for axis in fig.axes[2:]:
    axis.set_yscale("linear")
    axis.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    axis.axhline(0.0, color="#777777", linewidth=0.6)
    axis.set_title(axis.get_title() + " (circular residual)")
fig.savefig(OUTPUT / "transverse.png", dpi=180)
plt.close(fig)
display(Image(filename=str(OUTPUT / "transverse.png")))

summary = {
    "arcs_version": arcs.__version__, "case": case.to_dict(),
    "frequency_samples": int(frequency.size),
    "f_min_Hz": float(frequency[0]), "f_max_Hz": float(frequency[-1]),
    "pipe_length_m": solver.pipe_length_m,
    "longitudinal_units": solver.longitudinal_units,
    "dipolar_quadrupolar_units": solver.transverse_coefficient_units,
    "component": response.component, "mesh_report": report,
    "elapsed_s": elapsed_s, "frequency_parallelism": False,
    "automatic_accuracy_certification": False,
}
(OUTPUT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print("Saved:", ", ".join(p.name for p in sorted(OUTPUT.iterdir()) if p.is_file()))

## 7. The one-command workflow

From the extracted ARCS release root, the existing runner performs these steps:

```bash
python examples/run_case.py examples/geometries/bare_copper.json --output results/my_pipe
```

It defaults to 10 Hz–10¹³ Hz, 100 frequencies, β from the file (default 1), 1.0 m
pipe totals, a shared minimum of 256 panels, and longitudinal plus transverse results.
For a saved GUI model, replace the geometry path with `my_pipe.json`.

An explicit larger sweep is:

```bash
python examples/run_case.py my_pipe.json --f-min 10 --f-max 1e13 --points 1000 --panels 384 --pipe-length-m 1.0 --output results/fine_sweep
```

`--longitudinal-only` avoids transverse calculations; `--per-unit-length` selects
per-length normalization. `--component resistive_wall` selects RW alone.
The runner saves `impedance.csv`, `impedance.png`, `transverse.csv`,
`transverse.png`, `geometry.png` and `summary.json`.

**Current mesh override behavior:** `--panels` changes the shared minimum, including
its default of 256. Saved per-boundary overrides still take precedence. In Python,
`case.make_solver()` preserves the saved settings; an explicit
`mesh={"minimum_panels": 384}` call override takes precedence on every boundary.
Always inspect actual panel counts. Changing this call override alone does not
change the case object's preview, so update/reload the case too when comparing
modified meshes visually.

## 8. Refinement and parallel computing

Three resolutions serve different purposes:

| Control | Resolves | Does not automatically improve |
|---|---|---|
| Number/location of frequency samples | Peaks, humps and zero crossings versus frequency | Boundary-field accuracy |
| Panel count, maximum panel length, curve tolerance and corner grading | Geometry and boundary-field variation | Frequency sampling |
| GUI/JSON Gauss points per panel | Geometry sampling/export in this release | Maxwell kernel integration order |

First scan the broad band, then compare successively refined meshes at selected
low, middle and high frequencies and around any sharp feature. Refine until changes
are acceptable for the quantities you need, especially the imaginary component.
Then run the final grid and check difficult regions again. A minimum panel count is
not an automatic error bound.

For a bare pipe the Maxwell matching matrix has roughly 4N rows/columns for N
aperture panels; layered cases also require interface matching. Dense linear
algebra gives a rough O(N²) memory and O(N³) factorization guide, with additional
kernel/integration costs. Increasing frequency count F increases work approximately
in proportion to F at fixed mesh. Doubling N can therefore be much more expensive
than doubling F. Actual timings depend on geometry, materials, frequency and BLAS.

Frequencies can be solved independently once the case is fixed, so process-based
frequency chunks are a useful future extension. **ARCS 0.6.4 has no `--jobs`, MPI,
GPU or frequency-worker option.** SciPy's BLAS may already use CPU threads for one
matrix solve, depending on the installation; that is not concurrent frequency solves.
Each process would need its own matrices/solver and appreciable memory. Limit BLAS
threads per worker to prevent oversubscription, return results with frequency
indices, and verify equality with serial results before adopting a parallel runner.
Geometry can be prepared once per worker rather than once per frequency.

Checkpoint/resume is also not built into the current runner; use separate output
directories for separate runs. Chunked output/checkpoints would be valuable alongside
future frequency parallelism. No such new solver features are introduced by this tutorial.

## 9. Impedance first, wake conversion afterward

The solved output here is impedance. For the user's DCT/DST wake conversion, construct
the uniform frequency grid and justified DC treatment required by that transform.
Do not feed this logarithmic 100-point scan directly into a DCT/DST. Frequency range,
spacing and transform window determine the wake's time resolution and range; they
are a separate study from boundary-mesh convergence.